# V9A-Fast — V7 Cached DINOv3 Embedding Head Zoo

Notebook ini mengeksplorasi banyak classification head langsung dari frozen embedding cache V7:

- neural heads,
- linear models,
- LightGBM,
- XGBoost,
- CatBoost,
- Extra Trees,
- prototype/k-NN,
- pairwise probability blends,
- binary Recyclable–Organic specialist.

## Competition safety

Notebook ini **train-only**. Model dan hyperparameter dipilih dari fold train yang sudah tersedia pada cache V7.

Aturan kompetisi menyatakan bahwa Data Uji hanya boleh digunakan untuk prediksi akhir dan tidak boleh digunakan untuk training, validation, model selection, atau hyperparameter tuning. Karena `manual-labelling.csv` berisi label Data Uji, notebook ini memasang hard guard agar file tersebut tidak pernah dibaca.

Tidak ada cell untuk:

- membaca gambar test,
- membaca label manual test,
- memilih model berdasarkan test,
- mengoreksi prediksi test,
- membuat submission.

Setelah recipe terbaik dipilih secara train-only, inference test harus dibuat dalam notebook terpisah.

## Experiment protocol

1. Temukan dan validasi `v7-results`.
2. Muat cache `[N, 4, 2, 4096]`.
3. Screen beberapa readout menggunakan development fold 0.
4. Jalankan head zoo pada readout terbaik.
5. Cari blend dan R/O specialist hanya pada development fold 0.
6. Kunci recipe.
7. Konfirmasi recipe pada folds 1–4.
8. Simpan leaderboard, OOF prediction, dan recipe final.

Gunakan `RUN_PROFILE="smoke"` untuk pengecekan cepat, `"balanced"` untuk eksperimen utama, dan `"full"` untuk pencarian lebih luas.

## Runtime note

Dense DINOv3 embeddings make tree and neighbor models expensive. This version uses staged search:

- linear and neural heads are tested across selected readouts,
- LightGBM and XGBoost are tested only on the best screened readout,
- ExtraTrees, k-NN, and CatBoost are omitted from the default `fast` profile,
- tree parameters are reduced to practical values.

Use `RUN_PROFILE="full"` only after the fast run identifies a promising model family.

In [2]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))
print("Compiled architectures:", torch.cuda.get_arch_list())

x = torch.randn(8, 8192, device="cuda")
y = torch.nn.functional.layer_norm(x, (8192,))
torch.cuda.synchronize()

print("CUDA test passed:", y.shape)

GPU: Tesla T4
Capability: (7, 5)
Compiled architectures: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
CUDA test passed: torch.Size([8, 8192])


In [3]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import RidgeClassifier, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# =========================
# USER CONFIGURATION
# =========================
RUN_PROFILE = "fast"   # "smoke", "fast", "balanced", or "full"
SEED = 42
DEVELOPMENT_FOLD = 0
ASSESSMENT_FOLDS = (1, 2, 3, 4)
N_FOLDS = 5
NUM_CLASSES = 3
CLASS_NAMES = ("Recyclable", "Electronic", "Organic")

EXPECTED_ROWS = 26_527
EXPECTED_SHAPE = (26_527, 4, 2, 4096)
EXPECTED_CACHE_SIGNATURE = (
    "c722dc39a8cba286d3aba34d88f3e8f87071353bc6585e755873659de8560ddc"
)
EXPECTED_MODEL_ID = "facebook/dinov3-vit7b16-pretrain-lvd1689m"

OUTPUT_DIR = Path("/kaggle/working/v9a_v7_cached_head_zoo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = OUTPUT_DIR / "predictions"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

RESUME = False
SAVE_ALL_MODELS = False
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

PROFILE = {
    "smoke": {
        "top_readouts": 1,
        "head_names": ["sgd_log", "torch_mlp512", "lightgbm"],
        "blend_pool": 3,
        "assessment_recipes": 1,
        "torch_epochs": 6,
        "specialists": ["lightgbm"],
    },
    "fast": {
        "top_readouts": 2,
        "head_names": [
            "sgd_log",
            "ridge",
            "linear_svc",
            "torch_linear",
            "torch_mlp512",
            "torch_msd_mlp512",
            "torch_residual512",
            "torch_cosine",
            "lightgbm",
            "xgboost",
        ],
        "blend_pool": 6,
        "assessment_recipes": 3,
        "torch_epochs": 16,
        "specialists": ["lightgbm", "torch_mlp512"],
    },
    "balanced": {
        "top_readouts": 2,
        "head_names": [
            "sgd_log",
            "ridge",
            "linear_svc",
            "torch_linear",
            "torch_mlp256",
            "torch_mlp512",
            "torch_msd_mlp512",
            "torch_residual512",
            "torch_cosine",
            "lightgbm",
            "xgboost",
        ],
        "blend_pool": 6,
        "assessment_recipes": 3,
        "torch_epochs": 20,
        "specialists": ["lightgbm", "xgboost", "torch_mlp512"],
    },
    "full": {
        "top_readouts": 4,
        "head_names": [
            "sgd_log",
            "ridge",
            "linear_svc",
            "nearest_centroid",
            "knn_cosine",
            "extra_trees",
            "torch_linear",
            "torch_mlp256",
            "torch_mlp512",
            "torch_mlp1024",
            "torch_msd_linear",
            "torch_msd_mlp512",
            "torch_residual512",
            "torch_cosine",
            "lightgbm",
            "xgboost",
            "catboost",
        ],
        "blend_pool": 10,
        "assessment_recipes": 5,
        "torch_epochs": 30,
        "specialists": [
            "lightgbm",
            "xgboost",
            "catboost",
            "torch_mlp512",
            "torch_msd_mlp512",
        ],
    },
}[RUN_PROFILE]

EXPENSIVE_MODELS = {
    "extra_trees",
    "lightgbm",
    "xgboost",
    "catboost",
    "knn_cosine",
}
MAX_EXPENSIVE_READOUTS = 1
TREE_MAX_DIMENSION = 8192

print("Profile:", RUN_PROFILE)
print("Device:", DEVICE)
print("Output:", OUTPUT_DIR)

Profile: fast
Device: cuda:0
Output: /kaggle/working/v9a_v7_cached_head_zoo


In [4]:
# ============================================================
# HARD GUARD: NEVER LOAD MANUAL TEST LABELS DURING COMPETITION
# ============================================================

KAGGLE_INPUT = Path("/kaggle/input")
if not KAGGLE_INPUT.exists():
    raise RuntimeError("This notebook is designed for Kaggle: /kaggle/input was not found.")

blocked_files = []
for path in KAGGLE_INPUT.rglob("*.csv"):
    normalized = path.name.lower().replace("_", "-")
    if (
        normalized in {"manual-labelling.csv", "manual-labeling.csv"}
        or ("manual" in normalized and ("labell" in normalized or "label" in normalized))
    ):
        blocked_files.append(path)

if blocked_files:
    raise RuntimeError(
        "Competition safety guard triggered. Remove manual test-label files from notebook inputs:\n"
        + "\n".join(str(path) for path in blocked_files)
    )

print("Safety guard passed: no manual test-label CSV is attached.")

Safety guard passed: no manual test-label CSV is attached.


In [5]:
# =========================
# REPRODUCIBILITY HELPERS
# =========================

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def stable_json(value: Any) -> str:
    return json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)


def spec_key(spec: dict[str, Any]) -> str:
    return hashlib.sha256(stable_json(spec).encode()).hexdigest()[:16]


def atomic_json_dump(value: Any, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, default=str))
    temporary.replace(path)


def stable_softmax(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])
    shifted = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    return probabilities.astype(np.float32)


def clear_memory(*objects: Any) -> None:
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


seed_everything()

In [6]:
# =========================
# LOCATE V7 ARTIFACT ROOT
# =========================

manifest_candidates = []
for manifest_path in KAGGLE_INPUT.rglob("feature_cache_manifest.json"):
    candidate_root = manifest_path.parent
    if (candidate_root / "feature_cache" / "shards").is_dir():
        manifest_candidates.append(candidate_root)

if not manifest_candidates:
    raise FileNotFoundError(
        "No V7 artifact root was found. Attach the Kaggle dataset riantamaputra/v7-results."
    )

validated_candidates = []
for root in manifest_candidates:
    try:
        manifest = json.loads((root / "feature_cache_manifest.json").read_text())
    except Exception:
        continue
    if (
        manifest.get("complete") is True
        and tuple(manifest.get("feature_tensor_shape", [])) == EXPECTED_SHAPE
        and manifest.get("cache_signature") == EXPECTED_CACHE_SIGNATURE
    ):
        validated_candidates.append(root)

if len(validated_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one compatible V7 root, found:\n"
        + "\n".join(str(path) for path in validated_candidates)
    )

V7_ROOT = validated_candidates[0]
FEATURE_CACHE_DIR = V7_ROOT / "feature_cache" / "shards"
MANIFEST_PATH = V7_ROOT / "feature_cache_manifest.json"

print("V7 root:", V7_ROOT)
print("Cache shards:", FEATURE_CACHE_DIR)

V7 root: /kaggle/input/datasets/riantamaputra/v7-results/v7_dinov3_vit7b_frozen_raw
Cache shards: /kaggle/input/datasets/riantamaputra/v7-results/v7_dinov3_vit7b_frozen_raw/feature_cache/shards


In [7]:
# =========================
# VALIDATE MANIFEST
# =========================

manifest = json.loads(MANIFEST_PATH.read_text())
cache_identity = manifest["cache_identity"]

assert manifest["complete"] is True
assert manifest["rows"] == EXPECTED_ROWS
assert tuple(manifest["feature_tensor_shape"]) == EXPECTED_SHAPE
assert manifest["cache_signature"] == EXPECTED_CACHE_SIGNATURE
assert cache_identity["model_id"] == EXPECTED_MODEL_ID
assert cache_identity["image_size"] == 256
assert cache_identity["last_four_block_indices"] == [36, 37, 38, 39]
assert cache_identity["pooled_layout"] == ["cls", "mean_true_patch_tokens"]
assert cache_identity["feature_storage_dtype"] == "float16"

print(json.dumps({
    "rows": manifest["rows"],
    "shape": manifest["feature_tensor_shape"],
    "shards": manifest["shard_count"],
    "model": cache_identity["model_id"],
    "image_size": cache_identity["image_size"],
    "blocks": cache_identity["last_four_block_indices"],
    "pooling": cache_identity["pooled_layout"],
    "signature": manifest["cache_signature"],
}, indent=2))

{
  "rows": 26527,
  "shape": [
    26527,
    4,
    2,
    4096
  ],
  "shards": 104,
  "model": "facebook/dinov3-vit7b16-pretrain-lvd1689m",
  "image_size": 256,
  "blocks": [
    36,
    37,
    38,
    39
  ],
  "pooling": [
    "cls",
    "mean_true_patch_tokens"
  ],
  "signature": "c722dc39a8cba286d3aba34d88f3e8f87071353bc6585e755873659de8560ddc"
}


In [8]:
# =========================
# LOAD AND AUDIT ALL SHARDS
# =========================

shard_paths = sorted(FEATURE_CACHE_DIR.glob("train_*.pt"))
if len(shard_paths) != manifest["shard_count"]:
    raise RuntimeError(
        f"Shard count mismatch: found {len(shard_paths)}, expected {manifest['shard_count']}."
    )

cached_features = torch.empty(EXPECTED_SHAPE, dtype=torch.float16)
cached_labels = torch.empty(EXPECTED_ROWS, dtype=torch.long)
cached_folds = torch.empty(EXPECTED_ROWS, dtype=torch.long)
image_ids: list[str | None] = [None] * EXPECTED_ROWS

occupied = np.zeros(EXPECTED_ROWS, dtype=bool)

started = time.perf_counter()
for shard_index, shard_path in enumerate(shard_paths):
    payload = torch.load(shard_path, map_location="cpu", weights_only=False)
    metadata = payload["metadata"]
    start = int(metadata["start"])
    end = int(metadata["end"])

    features = payload["features"]
    labels = payload["labels"]
    folds = payload["folds"]
    ids = payload["image_ids"]

    expected_rows = end - start
    if tuple(features.shape) != (expected_rows, 4, 2, 4096):
        raise RuntimeError(f"Invalid feature shape in {shard_path}: {tuple(features.shape)}")
    if features.dtype != torch.float16:
        raise RuntimeError(f"Invalid dtype in {shard_path}: {features.dtype}")
    if not torch.isfinite(features).all().item():
        raise RuntimeError(f"NaN/Inf found in {shard_path}")
    if occupied[start:end].any():
        raise RuntimeError(f"Overlapping shard range: {shard_path}")

    cached_features[start:end].copy_(features)
    cached_labels[start:end].copy_(labels)
    cached_folds[start:end].copy_(folds)
    image_ids[start:end] = list(ids)
    occupied[start:end] = True

    if (shard_index + 1) % 20 == 0 or shard_index + 1 == len(shard_paths):
        print(f"Loaded {shard_index + 1:03d}/{len(shard_paths)} shards")

if not occupied.all():
    raise RuntimeError("Some cache rows were not filled.")
if any(image_id is None for image_id in image_ids):
    raise RuntimeError("Some image IDs are missing.")
if not torch.isfinite(cached_features).all().item():
    raise RuntimeError("Assembled cache contains NaN/Inf.")

print("Loaded:", tuple(cached_features.shape), cached_features.dtype)
print("Elapsed seconds:", round(time.perf_counter() - started, 2))
print("Label counts:", torch.bincount(cached_labels, minlength=NUM_CLASSES).tolist())
print("Fold counts:", torch.bincount(cached_folds, minlength=N_FOLDS).tolist())

Loaded 020/104 shards
Loaded 040/104 shards
Loaded 060/104 shards
Loaded 080/104 shards
Loaded 100/104 shards
Loaded 104/104 shards
Loaded: (26527, 4, 2, 4096) torch.float16
Elapsed seconds: 20.13
Label counts: [9999, 3961, 12567]
Fold counts: [5306, 5306, 5305, 5305, 5305]


In [9]:
# =========================
# DATA CONTRACT
# =========================

EXPECTED_CLASS_COUNTS = [9_999, 3_961, 12_567]
if torch.bincount(cached_labels, minlength=NUM_CLASSES).tolist() != EXPECTED_CLASS_COUNTS:
    raise RuntimeError("Canonical class counts changed.")

fold_class_table = pd.crosstab(
    pd.Series(cached_folds.numpy(), name="fold"),
    pd.Series(cached_labels.numpy(), name="label"),
)
display(fold_class_table)

data_contract = {
    "cache_signature": manifest["cache_signature"],
    "rows": EXPECTED_ROWS,
    "feature_shape": list(EXPECTED_SHAPE),
    "class_counts": EXPECTED_CLASS_COUNTS,
    "fold_class_counts": fold_class_table.to_dict(),
    "development_fold": DEVELOPMENT_FOLD,
    "assessment_folds": list(ASSESSMENT_FOLDS),
    "manual_test_labels_accessed": False,
    "test_images_accessed": False,
}
atomic_json_dump(data_contract, OUTPUT_DIR / "data_contract.json")

label,0,1,2
fold,,,
0,2000,792,2514
1,2000,792,2514
2,2000,792,2513
3,2000,792,2513
4,1999,793,2513


## Readout registry

Cache V7 menyimpan empat late transformer blocks. Pada setiap block tersedia:

- `CLS` global embedding,
- `mean_true_patch_tokens`.

Readout berikut dapat dibuat tanpa menjalankan DINOv3 lagi:

- final CLS,
- final patch mean,
- concatenation,
- difference,
- absolute difference,
- element-wise product,
- mean dua/four late blocks,
- layer mean and standard deviation,
- L2-normalized variants.

In [10]:
# =========================
# READOUT FACTORY
# =========================

BASE_READOUTS = (
    "final_cls",
    "final_patch",
    "final_concat",
    "final_sum",
    "final_diff",
    "final_absdiff",
    "final_product",
    "final_concat_diff",
    "mean_last2_concat",
    "mean_last4_concat",
    "layer_mean_std_concat",
)

READOUT_SCREEN = (
    "final_cls__l2",
    "final_patch__l2",
    "final_concat",
    "final_concat__l2",
    "final_sum__l2",
    "final_diff__l2",
    "final_absdiff__l2",
    "final_product__l2",
    "final_concat_diff",
    "mean_last2_concat",
    "mean_last4_concat",
    "layer_mean_std_concat",
)

def readout_dimension(name: str) -> int:
    base = name.replace("__l2", "")
    dimensions = {
        "final_cls": 4096,
        "final_patch": 4096,
        "final_concat": 8192,
        "final_sum": 4096,
        "final_diff": 4096,
        "final_absdiff": 4096,
        "final_product": 4096,
        "final_concat_diff": 12_288,
        "mean_last2_concat": 8192,
        "mean_last4_concat": 8192,
        "layer_mean_std_concat": 16_384,
    }
    return dimensions[base]


def _select_feature_slice(
    indices: np.ndarray | list[int],
    layer_slice: Any,
    representation_index: int,
) -> torch.Tensor:
    index_tensor = torch.as_tensor(np.asarray(indices, dtype=np.int64))
    source = cached_features[:, layer_slice, representation_index]
    return source.index_select(0, index_tensor).float()


def build_readout(name: str, indices: np.ndarray | list[int]) -> np.ndarray:
    normalized = name.endswith("__l2")
    base = name.replace("__l2", "")

    if base == "final_cls":
        output = _select_feature_slice(indices, -1, 0)

    elif base == "final_patch":
        output = _select_feature_slice(indices, -1, 1)

    elif base in {
        "final_concat",
        "final_sum",
        "final_diff",
        "final_absdiff",
        "final_product",
        "final_concat_diff",
    }:
        final_cls = _select_feature_slice(indices, -1, 0)
        final_patch = _select_feature_slice(indices, -1, 1)

        if base == "final_concat":
            output = torch.cat([final_cls, final_patch], dim=1)
        elif base == "final_sum":
            output = final_cls + final_patch
        elif base == "final_diff":
            output = final_cls - final_patch
        elif base == "final_absdiff":
            output = torch.abs(final_cls - final_patch)
        elif base == "final_product":
            output = final_cls * final_patch
        else:
            output = torch.cat(
                [final_cls, final_patch, final_cls - final_patch],
                dim=1,
            )
        del final_cls, final_patch

    elif base == "mean_last2_concat":
        cls_layers = _select_feature_slice(indices, slice(-2, None), 0)
        patch_layers = _select_feature_slice(indices, slice(-2, None), 1)
        output = torch.cat(
            [cls_layers.mean(dim=1), patch_layers.mean(dim=1)],
            dim=1,
        )
        del cls_layers, patch_layers

    elif base == "mean_last4_concat":
        cls_layers = _select_feature_slice(indices, slice(None), 0)
        patch_layers = _select_feature_slice(indices, slice(None), 1)
        output = torch.cat(
            [cls_layers.mean(dim=1), patch_layers.mean(dim=1)],
            dim=1,
        )
        del cls_layers, patch_layers

    elif base == "layer_mean_std_concat":
        cls_layers = _select_feature_slice(indices, slice(None), 0)
        patch_layers = _select_feature_slice(indices, slice(None), 1)
        output = torch.cat([
            cls_layers.mean(dim=1),
            cls_layers.std(dim=1, unbiased=False),
            patch_layers.mean(dim=1),
            patch_layers.std(dim=1, unbiased=False),
        ], dim=1)
        del cls_layers, patch_layers

    else:
        raise KeyError(f"Unknown readout: {name}")

    array = output.numpy().astype(np.float32, copy=False)
    if normalized:
        array = normalize(array, norm="l2", axis=1, copy=False).astype(np.float32)

    expected_shape = (len(indices), readout_dimension(name))
    if array.shape != expected_shape:
        raise RuntimeError(
            f"Readout shape mismatch for {name}: {array.shape}, expected {expected_shape}"
        )
    if not np.isfinite(array).all():
        raise RuntimeError(f"Readout contains NaN/Inf: {name}")

    del output
    gc.collect()
    return array


for name in READOUT_SCREEN:
    print(f"{name:26s} -> {readout_dimension(name):6d}")

final_cls__l2              ->   4096
final_patch__l2            ->   4096
final_concat               ->   8192
final_concat__l2           ->   8192
final_sum__l2              ->   4096
final_diff__l2             ->   4096
final_absdiff__l2          ->   4096
final_product__l2          ->   4096
final_concat_diff          ->  12288
mean_last2_concat          ->   8192
mean_last4_concat          ->   8192
layer_mean_std_concat      ->  16384


In [11]:
# =========================
# METRICS AND RESULT HELPERS
# =========================

def metric_bundle(y_true: np.ndarray, probabilities: np.ndarray) -> dict[str, Any]:
    y_true = np.asarray(y_true, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if probabilities.shape != (len(y_true), NUM_CLASSES):
        raise RuntimeError(f"Probability shape mismatch: {probabilities.shape}")
    if not np.isfinite(probabilities).all():
        raise RuntimeError("Probabilities contain NaN/Inf.")

    probabilities = probabilities / probabilities.sum(axis=1, keepdims=True)
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(y_true, predictions, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        y_true,
        predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=list(CLASS_NAMES),
        output_dict=True,
        zero_division=0,
    )

    return {
        "rows": int(len(y_true)),
        "macro_f1": float(f1_score(y_true, predictions, average="macro")),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "log_loss": float(log_loss(y_true, probabilities, labels=list(range(NUM_CLASSES)))),
        "errors": int((predictions != y_true).sum()),
        "f1_recyclable": float(report["Recyclable"]["f1-score"]),
        "f1_electronic": float(report["Electronic"]["f1-score"]),
        "f1_organic": float(report["Organic"]["f1-score"]),
        "recyclable_to_organic": int(matrix[0, 2]),
        "organic_to_recyclable": int(matrix[2, 0]),
        "ro_errors": int(matrix[0, 2] + matrix[2, 0]),
        "confusion_matrix": matrix.tolist(),
        "classification_report": report,
    }


def prediction_frame(
    indices: np.ndarray,
    probabilities: np.ndarray,
    recipe_name: str,
    fold: int,
    stage: str,
) -> pd.DataFrame:
    indices = np.asarray(indices, dtype=np.int64)
    y_true = cached_labels[torch.as_tensor(indices)].numpy()
    predictions = probabilities.argmax(axis=1)
    sorted_probabilities = np.sort(probabilities, axis=1)
    return pd.DataFrame({
        "row_index": indices,
        "image_id": np.asarray(image_ids, dtype=object)[indices],
        "label": y_true,
        "prediction": predictions,
        "prob_0": probabilities[:, 0],
        "prob_1": probabilities[:, 1],
        "prob_2": probabilities[:, 2],
        "confidence": sorted_probabilities[:, -1],
        "margin": sorted_probabilities[:, -1] - sorted_probabilities[:, -2],
        "correct": predictions == y_true,
        "fold": fold,
        "stage": stage,
        "recipe": recipe_name,
    })


def save_candidate_result(
    spec: dict[str, Any],
    indices: np.ndarray,
    probabilities: np.ndarray,
    metrics: dict[str, Any],
    stage: str,
    fold: int,
) -> None:
    key = spec_key(spec)
    stem = f"{stage}_fold{fold}_{key}"
    prediction_frame(indices, probabilities, key, fold, stage).to_csv(
        PRED_DIR / f"{stem}.csv", index=False
    )
    payload = {
        "key": key,
        "spec": spec,
        "stage": stage,
        "fold": fold,
        "metrics": metrics,
    }
    atomic_json_dump(payload, OUTPUT_DIR / f"{stem}.json")

## Neural head zoo

Semua neural heads melatih classifier saja. Frozen DINOv3 cache tidak berubah.

Candidate neural heads:

- linear probe,
- MLP hidden 256/512/1024,
- multi-sample dropout linear,
- multi-sample dropout MLP,
- residual MLP,
- cosine classifier,
- gated CLS–patch head untuk `final_concat`.

In [12]:
# =========================
# PYTORCH HEADS
# =========================

class LinearHead(nn.Module):
    def __init__(self, input_dim: int, n_classes: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


class MLPHead(nn.Module):
    def __init__(
        self,
        input_dim: int,
        n_classes: int,
        hidden_dim: int,
        dropout: float,
    ):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


class MultiSampleDropoutHead(nn.Module):
    def __init__(
        self,
        input_dim: int,
        n_classes: int,
        hidden_dim: int | None,
        dropout: float,
        samples: int = 5,
    ):
        super().__init__()
        self.samples = samples
        self.norm = nn.LayerNorm(input_dim)
        if hidden_dim is None:
            self.encoder = nn.Identity()
            encoded_dim = input_dim
        else:
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.GELU(),
            )
            encoded_dim = hidden_dim
        self.dropouts = nn.ModuleList([nn.Dropout(dropout) for _ in range(samples)])
        self.classifier = nn.Linear(encoded_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        encoded = self.encoder(self.norm(x))
        logits = [self.classifier(dropout(encoded)) for dropout in self.dropouts]
        return torch.stack(logits, dim=0).mean(dim=0)


class ResidualMLPHead(nn.Module):
    def __init__(
        self,
        input_dim: int,
        n_classes: int,
        hidden_dim: int = 512,
        dropout: float = 0.35,
    ):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.block = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = F.gelu(self.input_projection(self.norm(x)))
        hidden = hidden + self.block(hidden)
        return self.classifier(hidden)


class CosineHead(nn.Module):
    def __init__(self, input_dim: int, n_classes: int):
        super().__init__()
        self.class_weights = nn.Parameter(torch.empty(n_classes, input_dim))
        nn.init.normal_(self.class_weights, std=0.02)
        self.log_scale = nn.Parameter(torch.tensor(math.log(20.0)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = F.normalize(x, dim=1, eps=1e-6)
        weights = F.normalize(self.class_weights, dim=1, eps=1e-6)
        scale = self.log_scale.exp().clamp(1.0, 100.0)
        return scale * features @ weights.T


class GatedFinalHead(nn.Module):
    def __init__(self, n_classes: int, hidden_dim: int = 512, dropout: float = 0.30):
        super().__init__()
        self.cls_norm = nn.LayerNorm(4096)
        self.patch_norm = nn.LayerNorm(4096)
        self.cls_projection = nn.Linear(4096, hidden_dim)
        self.patch_projection = nn.Linear(4096, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(2 * hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] != 8192:
            raise RuntimeError("GatedFinalHead requires final_concat with 8192 dimensions.")
        cls_feature, patch_feature = x[:, :4096], x[:, 4096:]
        cls_hidden = self.dropout(F.gelu(self.cls_projection(self.cls_norm(cls_feature))))
        patch_hidden = self.dropout(F.gelu(self.patch_projection(self.patch_norm(patch_feature))))
        gate = torch.sigmoid(self.gate(torch.cat([cls_hidden, patch_hidden], dim=1)))
        fused = gate * cls_hidden + (1.0 - gate) * patch_hidden
        return self.classifier(fused)


def build_torch_head(
    name: str,
    input_dim: int,
    n_classes: int,
) -> tuple[nn.Module, float]:
    if name == "torch_linear":
        return LinearHead(input_dim, n_classes), 0.00
    if name == "torch_mlp256":
        return MLPHead(input_dim, n_classes, 256, 0.30), 0.03
    if name == "torch_mlp512":
        return MLPHead(input_dim, n_classes, 512, 0.40), 0.05
    if name == "torch_mlp1024":
        return MLPHead(input_dim, n_classes, 1024, 0.40), 0.05
    if name == "torch_msd_linear":
        return MultiSampleDropoutHead(input_dim, n_classes, None, 0.25, 5), 0.03
    if name == "torch_msd_mlp512":
        return MultiSampleDropoutHead(input_dim, n_classes, 512, 0.35, 5), 0.05
    if name == "torch_residual512":
        return ResidualMLPHead(input_dim, n_classes, 512, 0.35), 0.05
    if name == "torch_cosine":
        return CosineHead(input_dim, n_classes), 0.00
    if name == "torch_gated_final":
        return GatedFinalHead(n_classes, 512, 0.30), 0.03
    raise KeyError(name)


def train_torch_head(
    model_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    n_classes: int,
    fold: int,
    candidate_key: str,
    X_predict: np.ndarray | None = None,
) -> tuple[np.ndarray, dict[str, Any]]:
    seed_everything()
    input_dim = X_train.shape[1]
    if X_eval.shape[1] != input_dim:
        raise RuntimeError("Train/eval feature dimensions differ.")
    if X_predict is None:
        X_predict = X_eval
    if X_predict.shape[1] != input_dim:
        raise RuntimeError("Train/predict feature dimensions differ.")

    model, label_smoothing = build_torch_head(model_name, input_dim, n_classes)
    model = model.to(DEVICE)

    class_counts = np.bincount(y_train, minlength=n_classes)
    if (class_counts <= 0).any():
        raise RuntimeError("Training partition is missing a class.")
    class_weights = class_counts.sum() / (n_classes * class_counts)
    class_weights_tensor = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE,
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights_tensor,
        label_smoothing=label_smoothing,
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=8e-4,
        weight_decay=1e-4,
    )
    max_epochs = PROFILE["torch_epochs"]
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max_epochs,
    )

    batch_size = 128 if input_dim >= 16_384 else 256
    train_dataset = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train.astype(np.int64)),
    )
    eval_tensor = torch.from_numpy(X_eval)
    predict_tensor = torch.from_numpy(X_predict)

    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    best_state = None
    best_epoch = None
    best_f1 = -1.0
    best_loss = float("inf")
    stale = 0
    patience = 5
    history = []

    def predict_tensor_probabilities(tensor: torch.Tensor) -> np.ndarray:
        model.eval()
        logits_parts = []
        with torch.inference_mode():
            for start in range(0, len(tensor), 512):
                batch = tensor[start:start + 512].to(
                    DEVICE,
                    dtype=torch.float32,
                    non_blocking=True,
                )
                logits_parts.append(model(batch).cpu())
        logits = torch.cat(logits_parts).numpy()
        return stable_softmax(logits)

    started = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        seen = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(
                DEVICE,
                dtype=torch.float32,
                non_blocking=True,
            )
            batch_y = batch_y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss: {model_name}")
            loss.backward()
            gradient_norm = nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )
            if not torch.isfinite(gradient_norm):
                raise RuntimeError(f"Non-finite gradient: {model_name}")
            optimizer.step()

            total_loss += loss.item() * len(batch_y)
            seen += len(batch_y)

        scheduler.step()
        eval_probabilities = predict_tensor_probabilities(eval_tensor)
        eval_predictions = eval_probabilities.argmax(axis=1)
        eval_f1 = f1_score(y_eval, eval_predictions, average="macro")
        eval_loss = log_loss(
            y_eval,
            eval_probabilities,
            labels=list(range(n_classes)),
        )

        history.append({
            "epoch": epoch,
            "train_loss": total_loss / seen,
            "eval_macro_f1": eval_f1,
            "eval_log_loss": eval_loss,
        })
        print(
            f"{model_name} fold={fold} epoch={epoch:02d} "
            f"loss={total_loss / seen:.6f} f1={eval_f1:.6f}"
        )

        improved = (
            eval_f1 > best_f1 + 1e-8
            or (
                abs(eval_f1 - best_f1) <= 1e-8
                and eval_loss < best_loss
            )
        )
        if improved:
            best_f1 = float(eval_f1)
            best_loss = float(eval_loss)
            best_epoch = epoch
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    if best_state is None:
        raise RuntimeError(f"No checkpoint produced for {model_name}.")

    model.load_state_dict(best_state)
    probabilities = predict_tensor_probabilities(predict_tensor)
    metadata = {
        "best_epoch": best_epoch,
        "best_eval_macro_f1": best_f1,
        "best_eval_log_loss": best_loss,
        "training_seconds": time.perf_counter() - started,
        "parameter_count": sum(
            parameter.numel() for parameter in model.parameters()
        ),
        "label_smoothing": label_smoothing,
        "class_weights": class_weights.tolist(),
        "history": history,
    }

    if SAVE_ALL_MODELS:
        torch.save(
            {
                "model_name": model_name,
                "input_dim": input_dim,
                "n_classes": n_classes,
                "state_dict": best_state,
                "metadata": metadata,
            },
            MODEL_DIR / f"fold{fold}_{candidate_key}.pt",
        )

    del (
        model,
        train_loader,
        train_dataset,
        eval_tensor,
        predict_tensor,
        optimizer,
        scheduler,
    )
    clear_memory()
    return probabilities, metadata


## Classical head zoo

Classical models menerima frozen embedding sebagai tabular features. Package yang tidak tersedia akan dilewati secara eksplisit.

- SGD logistic regression,
- Ridge classifier,
- Linear SVM,
- cosine nearest centroid,
- cosine k-NN,
- Extra Trees,
- LightGBM,
- XGBoost,
- CatBoost.

In [13]:
# =========================
# OPTIONAL LIBRARY DISCOVERY
# =========================

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except Exception as exc:
    HAS_LIGHTGBM = False
    LIGHTGBM_ERROR = repr(exc)

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    XGBOOST_ERROR = repr(exc)

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception as exc:
    HAS_CATBOOST = False
    CATBOOST_ERROR = repr(exc)

print({
    "lightgbm": HAS_LIGHTGBM,
    "xgboost": HAS_XGBOOST,
    "catboost": HAS_CATBOOST,
})

{'lightgbm': True, 'xgboost': True, 'catboost': True}


In [14]:
# =========================
# CLASSICAL MODEL TRAINING
# =========================

def fit_predict_classical(
    model_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_valid: np.ndarray,
    n_classes: int,
) -> tuple[np.ndarray, dict[str, Any]]:
    started = time.perf_counter()

    if model_name == "nearest_centroid":
        normalized_train = normalize(X_train, norm="l2", axis=1)
        normalized_valid = normalize(X_valid, norm="l2", axis=1)
        prototypes = []
        for class_id in range(n_classes):
            prototype = normalized_train[y_train == class_id].mean(axis=0)
            prototype /= max(np.linalg.norm(prototype), 1e-12)
            prototypes.append(prototype)
        scores = normalized_valid @ np.stack(prototypes).T
        probabilities = stable_softmax(scores * 20.0)
        return probabilities, {
            "training_seconds": time.perf_counter() - started,
            "temperature": 20.0,
        }

    if model_name == "sgd_log":
        model = SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-4,
            max_iter=3000,
            tol=1e-4,
            class_weight="balanced",
            average=True,
            random_state=SEED,
            n_jobs=-1,
        )

    elif model_name == "ridge":
        model = RidgeClassifier(
            alpha=10.0,
            class_weight="balanced",
        )

    elif model_name == "linear_svc":
        model = LinearSVC(
            C=0.05,
            class_weight="balanced",
            dual="auto",
            max_iter=5000,
            random_state=SEED,
        )

    elif model_name == "knn_cosine":
        model = KNeighborsClassifier(
            n_neighbors=31,
            weights="distance",
            metric="cosine",
            n_jobs=-1,
        )

    elif model_name == "extra_trees":
        model = ExtraTreesClassifier(
            n_estimators=160,
            max_features="sqrt",
            max_depth=28,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=SEED,
            verbose=1,
        )

    elif model_name == "lightgbm":
        if not HAS_LIGHTGBM:
            raise RuntimeError(f"LightGBM unavailable: {LIGHTGBM_ERROR}")
        kwargs = dict(
            n_estimators=700,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=10,
            min_child_samples=40,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.10,
            reg_alpha=0.5,
            reg_lambda=2.0,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )
        if n_classes > 2:
            kwargs.update(objective="multiclass", num_class=n_classes)
        else:
            kwargs.update(objective="binary")
        model = LGBMClassifier(**kwargs)

    elif model_name == "xgboost":
        if not HAS_XGBOOST:
            raise RuntimeError(f"XGBoost unavailable: {XGBOOST_ERROR}")
        kwargs = dict(
            n_estimators=600,
            learning_rate=0.035,
            max_depth=4,
            min_child_weight=10,
            subsample=0.85,
            colsample_bytree=0.10,
            reg_alpha=0.5,
            reg_lambda=3.0,
            tree_method="hist",
            eval_metric="mlogloss" if n_classes > 2 else "logloss",
            random_state=SEED,
            n_jobs=-1,
        )
        if torch.cuda.is_available():
            kwargs["device"] = "cuda"
        if n_classes > 2:
            kwargs.update(
                objective="multi:softprob",
                num_class=n_classes,
            )
        else:
            kwargs.update(objective="binary:logistic")
        model = XGBClassifier(**kwargs)

    elif model_name == "catboost":
        if not HAS_CATBOOST:
            raise RuntimeError(f"CatBoost unavailable: {CATBOOST_ERROR}")
        kwargs = dict(
            loss_function="MultiClass" if n_classes > 2 else "Logloss",
            iterations=600,
            learning_rate=0.04,
            depth=6,
            l2_leaf_reg=5.0,
            random_strength=1.0,
            random_seed=SEED,
            verbose=100,
            allow_writing_files=False,
        )
        if torch.cuda.is_available():
            kwargs.update(task_type="GPU", devices="0")
        else:
            kwargs.update(task_type="CPU")
        model = CatBoostClassifier(**kwargs)

    else:
        raise KeyError(model_name)

    models_with_internal_class_weight = {
        "sgd_log",
        "ridge",
        "linear_svc",
        "extra_trees",
    }

    if model_name in models_with_internal_class_weight or model_name == "knn_cosine":
        model.fit(X_train, y_train)
    else:
        sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=y_train,
        )
        try:
            model.fit(X_train, y_train, sample_weight=sample_weight)
        except TypeError:
            model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_valid)
    else:
        probabilities = stable_softmax(model.decision_function(X_valid))

    probabilities = np.asarray(probabilities, dtype=np.float32)
    if probabilities.ndim == 1:
        probabilities = np.column_stack(
            [1.0 - probabilities, probabilities]
        )
    probabilities /= probabilities.sum(axis=1, keepdims=True)

    metadata = {
        "training_seconds": time.perf_counter() - started,
        "model_class": type(model).__name__,
    }

    if SAVE_ALL_MODELS:
        import joblib
        joblib.dump(
            model,
            MODEL_DIR / f"{model_name}_{int(time.time())}.joblib",
        )

    del model
    clear_memory()
    return probabilities, metadata


In [15]:
# ==================================================
# GENERIC STANDALONE CANDIDATE FIT/PREDICT
# ==================================================

def fit_predict_standalone(
    spec: dict[str, Any],
    train_indices: np.ndarray,
    valid_indices: np.ndarray,
    fold: int,
) -> tuple[np.ndarray, dict[str, Any]]:
    if spec["type"] != "standalone":
        raise ValueError("Expected standalone spec.")

    readout_name = spec["readout"]
    model_name = spec["model"]
    n_classes = int(spec.get("n_classes", NUM_CLASSES))

    X_train = build_readout(readout_name, train_indices)
    X_valid = build_readout(readout_name, valid_indices)
    y_train = cached_labels[torch.as_tensor(train_indices)].numpy()
    y_valid = cached_labels[torch.as_tensor(valid_indices)].numpy()

    key = spec_key(spec)
    candidate_started = time.perf_counter()
    print(
        f"\nCandidate {key}: model={model_name}, readout={readout_name}, "
        f"train={X_train.shape}, valid={X_valid.shape}"
    )

    if model_name.startswith("torch_"):
        probabilities, metadata = train_torch_head(
            model_name,
            X_train,
            y_train,
            X_valid,
            y_valid,
            n_classes=n_classes,
            fold=fold,
            candidate_key=key,
        )
    else:
        probabilities, metadata = fit_predict_classical(
            model_name,
            X_train,
            y_train,
            X_valid,
            n_classes=n_classes,
        )

    del X_train, X_valid, y_train, y_valid
    clear_memory()
    return probabilities, metadata

# Stage A — Readout screening

Readout dipilih hanya menggunakan fold 0. Screening memakai SGD logistic head yang sama agar perbandingan readout tidak tercampur perubahan model.

In [16]:
# =========================
# DEVELOPMENT PARTITION
# =========================

fold_array = cached_folds.numpy()
development_train_indices = np.flatnonzero(fold_array != DEVELOPMENT_FOLD)
development_valid_indices = np.flatnonzero(fold_array == DEVELOPMENT_FOLD)
development_labels = cached_labels[torch.as_tensor(development_valid_indices)].numpy()

print("Development train rows:", len(development_train_indices))
print("Development valid rows:", len(development_valid_indices))

Development train rows: 21221
Development valid rows: 5306


In [17]:
# =========================
# READOUT SCREEN
# =========================

readout_screen_rows = []

for readout_name in READOUT_SCREEN:
    spec = {
        "type": "standalone",
        "model": "sgd_log",
        "readout": readout_name,
        "purpose": "readout_screen",
    }
    key = spec_key(spec)
    result_path = OUTPUT_DIR / f"readout_screen_{key}.json"
    prediction_path = PRED_DIR / f"readout_screen_{key}.npy"

    if RESUME and result_path.exists() and prediction_path.exists():
        payload = json.loads(result_path.read_text())
        probabilities = np.load(prediction_path)
        metrics = payload["metrics"]
        metadata = payload["metadata"]
        print("Reused:", readout_name, metrics["macro_f1"])
    else:
        probabilities, metadata = fit_predict_standalone(
            spec,
            development_train_indices,
            development_valid_indices,
            DEVELOPMENT_FOLD,
        )
        metrics = metric_bundle(development_labels, probabilities)
        np.save(prediction_path, probabilities)
        atomic_json_dump(
            {"spec": spec, "metrics": metrics, "metadata": metadata},
            result_path,
        )

    readout_screen_rows.append({
        "key": key,
        "readout": readout_name,
        "dimension": readout_dimension(readout_name),
        **{k: v for k, v in metrics.items() if not isinstance(v, (dict, list))},
    })

readout_screen_df = (
    pd.DataFrame(readout_screen_rows)
    .sort_values(["macro_f1", "errors"], ascending=[False, True])
    .reset_index(drop=True)
)
readout_screen_df.to_csv(OUTPUT_DIR / "readout_screen.csv", index=False)
display(readout_screen_df)

selected_readouts = readout_screen_df.head(PROFILE["top_readouts"])["readout"].tolist()
if "final_concat" not in selected_readouts:
    selected_readouts.append("final_concat")

selected_readouts = list(dict.fromkeys(selected_readouts))
print("Selected readouts:", selected_readouts)


Candidate 56510518a3fa7085: model=sgd_log, readout=final_cls__l2, train=(21221, 4096), valid=(5306, 4096)

Candidate e6caf3ca98c4b294: model=sgd_log, readout=final_patch__l2, train=(21221, 4096), valid=(5306, 4096)

Candidate 2914bd13c27211ad: model=sgd_log, readout=final_concat, train=(21221, 8192), valid=(5306, 8192)

Candidate 2254c9f290025dfa: model=sgd_log, readout=final_concat__l2, train=(21221, 8192), valid=(5306, 8192)

Candidate 6ba2aa794764175a: model=sgd_log, readout=final_sum__l2, train=(21221, 4096), valid=(5306, 4096)

Candidate 89a69ef86e656e63: model=sgd_log, readout=final_diff__l2, train=(21221, 4096), valid=(5306, 4096)

Candidate 2a54c3475f63a7da: model=sgd_log, readout=final_absdiff__l2, train=(21221, 4096), valid=(5306, 4096)

Candidate 8d56e3c6a4024d92: model=sgd_log, readout=final_product__l2, train=(21221, 4096), valid=(5306, 4096)

Candidate 63d03feefef32fb9: model=sgd_log, readout=final_concat_diff, train=(21221, 12288), valid=(5306, 12288)

Candidate f1dac85

,key,readout,dimension,rows,macro_f1,accuracy,log_loss,errors,f1_recyclable,f1_electronic,f1_organic,recyclable_to_organic,organic_to_recyclable,ro_errors
0,3944aceabc08ac3e,mean_last4_concat,8192,5306,0.986890,0.984546,0.132961,82,0.979806,0.996207,0.984658,33,43,76
1,78cc6857620ba996,layer_mean_std_concat,16384,5306,0.986616,0.984357,0.138272,83,0.979826,0.995578,0.984444,31,45,76
2,2914bd13c27211ad,final_concat,8192,5306,0.985872,0.983415,0.108069,88,0.978607,0.995578,0.983430,31,50,81
3,63d03feefef32fb9,final_concat_diff,12288,5306,0.985322,0.983227,0.147594,89,0.978850,0.993679,0.983436,31,48,79
4,f1dac854eaf971a5,mean_last2_concat,8192,5306,0.985266,0.983038,0.133936,90,0.977844,0.994315,0.983639,33,48,81
5,6ba2aa794764175a,final_sum__l2,4096,5306,0.981338,0.979269,0.110535,110,0.973887,0.989899,0.980228,37,57,94
6,2254c9f290025dfa,final_concat__l2,8192,5306,0.980898,0.978892,0.114110,112,0.973403,0.989274,0.980016,36,59,95
7,56510518a3fa7085,final_cls__l2,4096,5306,0.980282,0.977761,0.115740,118,0.972000,0.990512,0.978334,52,51,103
8,89a69ef86e656e63,final_diff__l2,4096,5306,0.975047,0.972107,0.142450,148,0.965997,0.986767,0.972378,48,79,127
9,e6caf3ca98c4b294,final_patch__l2,4096,5306,0.974689,0.971353,0.135350,152,0.964136,0.988043,0.971888,43,90,133


Selected readouts: ['mean_last4_concat', 'layer_mean_std_concat', 'final_concat']


# Stage B — Head zoo on selected readouts

Setiap candidate dilatih pada folds 1–4 dan dinilai pada fold 0. Candidate yang gagal karena package atau resource dicatat sebagai `failed`, bukan disembunyikan.

In [18]:
# =========================
# STAGED HEAD CANDIDATE REGISTRY
# =========================

ranked_readouts = readout_screen_df["readout"].tolist()
best_expensive_readouts = ranked_readouts[:MAX_EXPENSIVE_READOUTS]

cheap_models = [
    model_name
    for model_name in PROFILE["head_names"]
    if model_name not in EXPENSIVE_MODELS
]
expensive_models = [
    model_name
    for model_name in PROFILE["head_names"]
    if model_name in EXPENSIVE_MODELS
]

head_specs: list[dict[str, Any]] = []

# Cheap linear/neural heads run across all selected readouts.
for readout_name in selected_readouts:
    for model_name in cheap_models:
        head_specs.append({
            "type": "standalone",
            "model": model_name,
            "readout": readout_name,
            "purpose": "head_zoo",
        })

# Expensive heads run only on the best screened readout.
for readout_name in best_expensive_readouts:
    dimension = readout_dimension(readout_name)
    if dimension > TREE_MAX_DIMENSION:
        print(
            "Skipping expensive models for",
            readout_name,
            "because dimension is",
            dimension,
        )
        continue

    for model_name in expensive_models:
        head_specs.append({
            "type": "standalone",
            "model": model_name,
            "readout": readout_name,
            "purpose": "head_zoo",
        })

# Always preserve V7-compatible neural controls.
for model_name in ("torch_mlp512", "torch_msd_mlp512"):
    head_specs.append({
        "type": "standalone",
        "model": model_name,
        "readout": "final_concat",
        "purpose": "head_zoo",
    })

head_specs.append({
    "type": "standalone",
    "model": "torch_gated_final",
    "readout": "final_concat",
    "purpose": "head_zoo",
})

deduplicated_specs = []
seen_keys = set()
for spec in head_specs:
    key = spec_key(spec)
    if key not in seen_keys:
        seen_keys.add(key)
        deduplicated_specs.append(spec)
head_specs = deduplicated_specs

candidate_table = pd.DataFrame([
    {
        "key": spec_key(spec),
        "model": spec["model"],
        "readout": spec["readout"],
        "dimension": readout_dimension(spec["readout"]),
        "expensive": spec["model"] in EXPENSIVE_MODELS,
    }
    for spec in head_specs
])

print("Selected readouts:", selected_readouts)
print("Expensive-model readouts:", best_expensive_readouts)
print("Head candidates:", len(head_specs))
display(candidate_table)

Selected readouts: ['mean_last4_concat', 'layer_mean_std_concat', 'final_concat']
Expensive-model readouts: ['mean_last4_concat']
Head candidates: 27


,key,model,readout,dimension,expensive
0,4fb730fbe70e964f,sgd_log,mean_last4_concat,8192,False
1,032d07893601130b,ridge,mean_last4_concat,8192,False
2,cb969df173cb635b,linear_svc,mean_last4_concat,8192,False
3,bf342c403abf1e27,torch_linear,mean_last4_concat,8192,False
4,dceb88081c8e8773,torch_mlp512,mean_last4_concat,8192,False
5,8ed362eb569205b9,torch_msd_mlp512,mean_last4_concat,8192,False
6,48fb3c92a840397a,torch_residual512,mean_last4_concat,8192,False
7,d0e7fe7416833908,torch_cosine,mean_last4_concat,8192,False
8,40a1efcbee2ee154,sgd_log,layer_mean_std_concat,16384,False
9,b0cda6d21ed0ef69,ridge,layer_mean_std_concat,16384,False


In [19]:
# =========================
# RUN HEAD ZOO
# =========================

development_results: list[dict[str, Any]] = []
development_probabilities: dict[str, np.ndarray] = {}
development_specs: dict[str, dict[str, Any]] = {}

for candidate_number, spec in enumerate(head_specs, start=1):
    key = spec_key(spec)
    result_path = OUTPUT_DIR / f"development_{key}.json"
    prediction_path = PRED_DIR / f"development_{key}.npy"

    print(f"\n[{candidate_number}/{len(head_specs)}] {key} {spec}")

    if RESUME and result_path.exists() and prediction_path.exists():
        payload = json.loads(result_path.read_text())
        if payload.get("status") == "success":
            probabilities = np.load(prediction_path)
            metrics = payload["metrics"]
            metadata = payload["metadata"]
            print("Reused score:", metrics["macro_f1"])
        else:
            development_results.append(payload)
            continue
    else:
        try:
            probabilities, metadata = fit_predict_standalone(
                spec,
                development_train_indices,
                development_valid_indices,
                DEVELOPMENT_FOLD,
            )
            metrics = metric_bundle(development_labels, probabilities)
            np.save(prediction_path, probabilities)
            payload = {
                "status": "success",
                "key": key,
                "spec": spec,
                "metrics": metrics,
                "metadata": metadata,
            }
            atomic_json_dump(payload, result_path)
        except Exception as exc:
            payload = {
                "status": "failed",
                "key": key,
                "spec": spec,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
            atomic_json_dump(payload, result_path)
            development_results.append(payload)
            print("FAILED:", repr(exc))
            clear_memory()
            continue

    development_probabilities[key] = probabilities
    development_specs[key] = spec
    development_results.append({
        "status": "success",
        "key": key,
        "spec": spec,
        "metrics": metrics,
        "metadata": metadata,
    })

successful_rows = []
failed_rows = []

for result in development_results:
    if result["status"] == "success":
        metrics = result["metrics"]
        successful_rows.append({
            "key": result["key"],
            "model": result["spec"]["model"],
            "readout": result["spec"]["readout"],
            "dimension": readout_dimension(result["spec"]["readout"]),
            **{k: v for k, v in metrics.items() if not isinstance(v, (dict, list))},
            "training_seconds": result["metadata"].get("training_seconds"),
        })
    else:
        failed_rows.append({
            "key": result["key"],
            "model": result["spec"]["model"],
            "readout": result["spec"]["readout"],
            "error": result["error"],
        })

development_leaderboard = (
    pd.DataFrame(successful_rows)
    .sort_values(["macro_f1", "errors"], ascending=[False, True])
    .reset_index(drop=True)
)
development_leaderboard.to_csv(
    OUTPUT_DIR / "development_standalone_leaderboard.csv", index=False
)
display(development_leaderboard)

if failed_rows:
    failed_df = pd.DataFrame(failed_rows)
    failed_df.to_csv(OUTPUT_DIR / "failed_candidates.csv", index=False)
    display(failed_df)


[1/27] 4fb730fbe70e964f {'type': 'standalone', 'model': 'sgd_log', 'readout': 'mean_last4_concat', 'purpose': 'head_zoo'}

Candidate 4fb730fbe70e964f: model=sgd_log, readout=mean_last4_concat, train=(21221, 8192), valid=(5306, 8192)

[2/27] 032d07893601130b {'type': 'standalone', 'model': 'ridge', 'readout': 'mean_last4_concat', 'purpose': 'head_zoo'}

Candidate 032d07893601130b: model=ridge, readout=mean_last4_concat, train=(21221, 8192), valid=(5306, 8192)

[3/27] cb969df173cb635b {'type': 'standalone', 'model': 'linear_svc', 'readout': 'mean_last4_concat', 'purpose': 'head_zoo'}

Candidate cb969df173cb635b: model=linear_svc, readout=mean_last4_concat, train=(21221, 8192), valid=(5306, 8192)

[4/27] bf342c403abf1e27 {'type': 'standalone', 'model': 'torch_linear', 'readout': 'mean_last4_concat', 'purpose': 'head_zoo'}

Candidate bf342c403abf1e27: model=torch_linear, readout=mean_last4_concat, train=(21221, 8192), valid=(5306, 8192)
torch_linear fold=0 epoch=01 loss=0.084775 f1=0.9838

,key,model,readout,dimension,rows,macro_f1,accuracy,log_loss,errors,f1_recyclable,f1_electronic,f1_organic,recyclable_to_organic,organic_to_recyclable,ro_errors,training_seconds
0,8ed362eb569205b9,torch_msd_mlp512,mean_last4_concat,8192,5306,0.990491,0.988881,0.094632,59,0.985796,0.996849,0.988827,20,34,54,13.180415
1,8eff61d9aa46ceb6,torch_mlp512,layer_mean_std_concat,16384,5306,0.990170,0.988315,0.093743,62,0.984807,0.997478,0.988226,21,37,58,27.366078
2,dceb88081c8e8773,torch_mlp512,mean_last4_concat,8192,5306,0.989902,0.988315,0.095158,62,0.985075,0.996217,0.988414,18,38,56,13.303525
3,85a24bec6d8a75c5,torch_mlp512,final_concat,8192,5306,0.989768,0.988315,0.101366,62,0.985268,0.995589,0.988446,25,30,55,7.808914
4,5e568f93459356bc,torch_msd_mlp512,layer_mean_std_concat,16384,5306,0.989594,0.987750,0.095331,65,0.984309,0.996849,0.987625,22,38,60,24.402509
5,e03f9d1980422fa9,torch_msd_mlp512,final_concat,8192,5306,0.989306,0.987750,0.095899,65,0.984301,0.995589,0.988029,22,36,58,13.446105
6,efc513d6834ff7d6,torch_gated_final,final_concat,8192,5306,0.988427,0.986807,0.073574,70,0.983271,0.994956,0.987054,29,33,62,7.721278
7,61ab2f2726bba07e,torch_residual512,layer_mean_std_concat,16384,5306,0.988417,0.986619,0.102825,71,0.982866,0.995589,0.986795,18,46,64,18.153490
8,48fb3c92a840397a,torch_residual512,mean_last4_concat,8192,5306,0.988093,0.986242,0.095845,73,0.982054,0.995584,0.986640,27,39,66,13.609404
9,0ade905859c7fd1d,torch_residual512,final_concat,8192,5306,0.987773,0.985677,0.096475,76,0.981198,0.996217,0.985904,40,30,70,7.667944


# Stage C — Pairwise probability blending

Blend hanya memakai prediction pada development fold. Alpha dicari pada grid yang sudah ditentukan. Candidate dengan inductive bias berbeda dapat berguna walaupun standalone score lebih rendah.

In [20]:
# =========================
# PAIRWISE BLEND SEARCH
# =========================

blend_pool_keys = development_leaderboard.head(PROFILE["blend_pool"])["key"].tolist()
alpha_grid = np.round(np.arange(0.10, 1.00, 0.10), 2)

blend_rows = []
blend_recipes: dict[str, dict[str, Any]] = {}
blend_probabilities: dict[str, np.ndarray] = {}

for left_position, left_key in enumerate(blend_pool_keys):
    for right_key in blend_pool_keys[left_position + 1:]:
        left_prob = development_probabilities[left_key]
        right_prob = development_probabilities[right_key]

        disagreement = float(
            (left_prob.argmax(axis=1) != right_prob.argmax(axis=1)).mean()
        )
        if disagreement == 0:
            continue

        for alpha in alpha_grid:
            probabilities = (
                alpha * left_prob + (1.0 - alpha) * right_prob
            ).astype(np.float32)
            probabilities /= probabilities.sum(axis=1, keepdims=True)
            metrics = metric_bundle(development_labels, probabilities)

            recipe = {
                "type": "blend",
                "left": development_specs[left_key],
                "right": development_specs[right_key],
                "alpha_left": float(alpha),
                "purpose": "pairwise_blend",
            }
            key = spec_key(recipe)

            blend_rows.append({
                "key": key,
                "left_key": left_key,
                "right_key": right_key,
                "alpha_left": alpha,
                "disagreement_rate": disagreement,
                **{k: v for k, v in metrics.items() if not isinstance(v, (dict, list))},
            })
            blend_recipes[key] = recipe
            blend_probabilities[key] = probabilities

if blend_rows:
    blend_leaderboard = (
        pd.DataFrame(blend_rows)
        .sort_values(["macro_f1", "errors"], ascending=[False, True])
        .drop_duplicates(subset=["left_key", "right_key"], keep="first")
        .reset_index(drop=True)
    )
else:
    blend_leaderboard = pd.DataFrame(columns=[
        "key", "left_key", "right_key", "alpha_left",
        "disagreement_rate", "macro_f1", "errors",
    ])

blend_leaderboard.to_csv(
    OUTPUT_DIR / "development_blend_leaderboard.csv",
    index=False,
)
display(blend_leaderboard.head(20))

,key,left_key,right_key,alpha_left,disagreement_rate,rows,macro_f1,accuracy,log_loss,errors,f1_recyclable,f1_electronic,f1_organic,recyclable_to_organic,organic_to_recyclable,ro_errors
0,35e966fbb1c1dfd7,8ed362eb569205b9,85a24bec6d8a75c5,0.9,0.004146,5306,0.990491,0.988881,0.095031,59,0.985796,0.996849,0.988827,20,34,54
1,2b8161d1788b0756,8ed362eb569205b9,dceb88081c8e8773,0.7,0.001696,5306,0.990342,0.988692,0.094587,60,0.985551,0.996849,0.988625,20,35,55
2,45f9d06646bb925f,8ed362eb569205b9,e03f9d1980422fa9,0.7,0.003015,5306,0.990342,0.988692,0.094729,60,0.985551,0.996849,0.988625,20,35,55
3,cbc292d6bd324943,85a24bec6d8a75c5,5e568f93459356bc,0.9,0.004900,5306,0.990211,0.988692,0.100478,60,0.985768,0.996222,0.988643,24,30,54
4,8180b55511a924b2,dceb88081c8e8773,85a24bec6d8a75c5,0.5,0.004712,5306,0.990199,0.988692,0.097749,60,0.985558,0.996217,0.988822,19,35,54
5,70f80e2d5bd03bb6,8ed362eb569205b9,8eff61d9aa46ceb6,0.8,0.002073,5306,0.990193,0.988504,0.094212,61,0.985305,0.996849,0.988423,20,36,56
6,aff449bcc6dcfc07,8ed362eb569205b9,5e568f93459356bc,0.9,0.001885,5306,0.990193,0.988504,0.094550,61,0.985305,0.996849,0.988423,20,36,56
7,bf378440bf7ae784,85a24bec6d8a75c5,e03f9d1980422fa9,0.9,0.003769,5306,0.990068,0.988692,0.100644,60,0.985775,0.995589,0.988840,23,30,53
8,5f63b4f2d6bfbf42,8eff61d9aa46ceb6,dceb88081c8e8773,0.6,0.001885,5306,0.990043,0.988315,0.093965,62,0.985060,0.996849,0.988221,20,37,57
9,e65f2db573ebfec2,8eff61d9aa46ceb6,e03f9d1980422fa9,0.9,0.003958,5306,0.990043,0.988315,0.093757,62,0.985060,0.996849,0.988221,20,37,57


# Stage D — Recyclable–Organic specialist

Specialist dilatih hanya pada label 0 dan 2 dari **train partition**. Probability Electronic dari main classifier tidak diubah. Specialist hanya membagi ulang probability mass Recyclable–Organic.

Adoption guard pada fold 0:

- Macro F1 harus naik,
- total error tidak boleh bertambah,
- Electronic F1 tidak boleh turun lebih dari 0.0005.

In [21]:
# =========================
# BINARY R/O SPECIALIST
# =========================

def fit_predict_binary_specialist(
    model_name: str,
    readout_name: str,
    train_indices: np.ndarray,
    valid_indices: np.ndarray,
    fold: int,
) -> tuple[np.ndarray, dict[str, Any]]:
    train_labels_original = cached_labels[
        torch.as_tensor(train_indices)
    ].numpy()
    train_ro_mask = train_labels_original != 1
    ro_train_indices = train_indices[train_ro_mask]
    ro_train_labels = (
        cached_labels[torch.as_tensor(ro_train_indices)].numpy() == 2
    ).astype(np.int64)

    valid_labels_original = cached_labels[
        torch.as_tensor(valid_indices)
    ].numpy()
    valid_ro_mask = valid_labels_original != 1
    ro_eval_labels = (
        valid_labels_original[valid_ro_mask] == 2
    ).astype(np.int64)

    X_train = build_readout(readout_name, ro_train_indices)
    X_predict = build_readout(readout_name, valid_indices)
    X_eval = X_predict[valid_ro_mask]

    key = spec_key({
        "type": "binary_specialist",
        "model": model_name,
        "readout": readout_name,
    })

    if model_name.startswith("torch_"):
        binary_probabilities, metadata = train_torch_head(
            model_name,
            X_train,
            ro_train_labels,
            X_eval,
            ro_eval_labels,
            n_classes=2,
            fold=fold,
            candidate_key=key,
            X_predict=X_predict,
        )
    else:
        binary_probabilities, metadata = fit_predict_classical(
            model_name,
            X_train,
            ro_train_labels,
            X_predict,
            n_classes=2,
        )

    p_organic = binary_probabilities[:, 1]
    del X_train, X_eval, X_predict
    clear_memory()
    return p_organic, metadata


def fuse_ro_specialist(
    main_probabilities: np.ndarray,
    p_organic: np.ndarray,
    alpha: float,
) -> np.ndarray:
    main = np.asarray(main_probabilities, dtype=np.float32)
    ro_mass = main[:, 0] + main[:, 2]

    specialist = main.copy()
    specialist[:, 0] = ro_mass * (1.0 - p_organic)
    specialist[:, 2] = ro_mass * p_organic

    output = main.copy()
    output[:, [0, 2]] = (
        (1.0 - alpha) * main[:, [0, 2]]
        + alpha * specialist[:, [0, 2]]
    )
    output /= output.sum(axis=1, keepdims=True)
    return output


top_main_keys = development_leaderboard.head(min(3, len(development_leaderboard)))["key"].tolist()
specialist_readouts = (
    ["final_diff__l2", "final_product__l2"]
    if RUN_PROFILE in {"smoke", "fast"}
    else ["final_concat_diff", "final_diff__l2", "final_product__l2"]
)
specialist_alpha_grid = (0.25, 0.50, 0.75, 1.00)

specialist_rows = []
specialist_recipes: dict[str, dict[str, Any]] = {}
specialist_probabilities: dict[str, np.ndarray] = {}

binary_cache: dict[tuple[str, str], tuple[np.ndarray, dict[str, Any]]] = {}

for specialist_model in PROFILE["specialists"]:
    for specialist_readout in specialist_readouts:
        cache_key = (specialist_model, specialist_readout)
        try:
            if cache_key not in binary_cache:
                binary_cache[cache_key] = fit_predict_binary_specialist(
                    specialist_model,
                    specialist_readout,
                    development_train_indices,
                    development_valid_indices,
                    DEVELOPMENT_FOLD,
                )
            p_organic, specialist_metadata = binary_cache[cache_key]
        except Exception as exc:
            print(
                "Specialist failed:",
                specialist_model,
                specialist_readout,
                repr(exc),
            )
            continue

        for main_key in top_main_keys:
            main_prob = development_probabilities[main_key]
            main_metrics = metric_bundle(development_labels, main_prob)

            for alpha in specialist_alpha_grid:
                probabilities = fuse_ro_specialist(main_prob, p_organic, alpha)
                metrics = metric_bundle(development_labels, probabilities)

                adopted = (
                    metrics["macro_f1"] > main_metrics["macro_f1"]
                    and metrics["errors"] <= main_metrics["errors"]
                    and metrics["f1_electronic"]
                    >= main_metrics["f1_electronic"] - 0.0005
                )

                recipe = {
                    "type": "ro_specialist",
                    "main": development_specs[main_key],
                    "specialist_model": specialist_model,
                    "specialist_readout": specialist_readout,
                    "alpha": float(alpha),
                    "purpose": "ro_specialist",
                }
                key = spec_key(recipe)

                specialist_rows.append({
                    "key": key,
                    "main_key": main_key,
                    "specialist_model": specialist_model,
                    "specialist_readout": specialist_readout,
                    "alpha": alpha,
                    "adopted_by_guard": adopted,
                    "main_macro_f1": main_metrics["macro_f1"],
                    "main_errors": main_metrics["errors"],
                    **{k: v for k, v in metrics.items() if not isinstance(v, (dict, list))},
                })
                specialist_recipes[key] = recipe
                specialist_probabilities[key] = probabilities

if specialist_rows:
    specialist_leaderboard = (
        pd.DataFrame(specialist_rows)
        .sort_values(
            ["adopted_by_guard", "macro_f1", "errors"],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )
else:
    specialist_leaderboard = pd.DataFrame(columns=[
        "key", "main_key", "specialist_model",
        "specialist_readout", "alpha", "adopted_by_guard",
        "macro_f1", "errors",
    ])

specialist_leaderboard.to_csv(
    OUTPUT_DIR / "development_ro_specialist_leaderboard.csv",
    index=False,
)
display(specialist_leaderboard.head(30))

torch_mlp512 fold=0 epoch=01 loss=0.197620 f1=0.979399
torch_mlp512 fold=0 epoch=02 loss=0.143244 f1=0.983405
torch_mlp512 fold=0 epoch=03 loss=0.130983 f1=0.983174
torch_mlp512 fold=0 epoch=04 loss=0.126614 f1=0.982289
torch_mlp512 fold=0 epoch=05 loss=0.125208 f1=0.983176
torch_mlp512 fold=0 epoch=06 loss=0.123218 f1=0.983401
torch_mlp512 fold=0 epoch=07 loss=0.122662 f1=0.983178
torch_mlp512 fold=0 epoch=01 loss=0.221647 f1=0.973812
torch_mlp512 fold=0 epoch=02 loss=0.155998 f1=0.978037
torch_mlp512 fold=0 epoch=03 loss=0.141766 f1=0.978917
torch_mlp512 fold=0 epoch=04 loss=0.133349 f1=0.978239
torch_mlp512 fold=0 epoch=05 loss=0.129076 f1=0.978258
torch_mlp512 fold=0 epoch=06 loss=0.126516 f1=0.979149
torch_mlp512 fold=0 epoch=07 loss=0.125199 f1=0.979591
torch_mlp512 fold=0 epoch=08 loss=0.124438 f1=0.979601
torch_mlp512 fold=0 epoch=09 loss=0.123328 f1=0.979820
torch_mlp512 fold=0 epoch=10 loss=0.122703 f1=0.980046
torch_mlp512 fold=0 epoch=11 loss=0.122400 f1=0.978692
torch_mlp5

,key,main_key,specialist_model,specialist_readout,alpha,adopted_by_guard,main_macro_f1,main_errors,rows,macro_f1,accuracy,log_loss,errors,f1_recyclable,f1_electronic,f1_organic,recyclable_to_organic,organic_to_recyclable,ro_errors
0,adc8eb5301608bf2,8eff61d9aa46ceb6,lightgbm,final_product__l2,0.25,False,0.990170,62,5306,0.989873,0.987938,0.096504,64,0.984324,0.997478,0.987817,20,40,60
1,864cd05b4badf319,8ed362eb569205b9,torch_mlp512,final_diff__l2,0.25,False,0.990491,59,5306,0.989744,0.987938,0.096693,64,0.984562,0.996849,0.987822,21,38,59
2,3eb3e880bbde9981,8eff61d9aa46ceb6,lightgbm,final_diff__l2,0.25,False,0.990170,62,5306,0.989596,0.987750,0.094409,65,0.984324,0.996849,0.987615,20,40,60
3,86672c2e25d68886,8ed362eb569205b9,lightgbm,final_diff__l2,0.25,False,0.990491,59,5306,0.989593,0.987750,0.095475,65,0.984301,0.996849,0.987630,23,37,60
4,40a0788652280403,8eff61d9aa46ceb6,torch_mlp512,final_diff__l2,0.25,False,0.990170,62,5306,0.989573,0.987561,0.095756,66,0.983819,0.997478,0.987423,22,40,62
5,a128a62563d0b5df,8ed362eb569205b9,lightgbm,final_product__l2,0.25,False,0.990491,59,5306,0.989297,0.987373,0.097375,67,0.983827,0.996849,0.987215,21,41,62
6,5c59da32396883ff,8eff61d9aa46ceb6,torch_mlp512,final_product__l2,0.25,False,0.990170,62,5306,0.989297,0.987373,0.097179,67,0.983827,0.996849,0.987215,21,41,62
7,ee3bbda64fd0050a,8ed362eb569205b9,torch_mlp512,final_diff__l2,0.50,False,0.990491,59,5306,0.989147,0.987184,0.099960,68,0.983574,0.996849,0.987018,22,41,63
8,c4d77d31cbf33cd3,8ed362eb569205b9,torch_mlp512,final_product__l2,0.25,False,0.990491,59,5306,0.989147,0.987184,0.098144,68,0.983574,0.996849,0.987018,22,41,63
9,b9ed13adbe871c0f,dceb88081c8e8773,torch_mlp512,final_diff__l2,0.25,False,0.989902,62,5306,0.989006,0.987184,0.097343,68,0.983590,0.996217,0.987210,20,42,62


# Stage E — Lock recipes before assessment

Standalone, blend, dan specialist digabungkan menjadi satu development leaderboard. Hanya recipe teratas yang dibawa ke assessment folds 1–4.

Control V7-style `torch_mlp512 + final_concat` selalu ikut assessment agar perbandingan tetap memiliki baseline.

In [22]:
# =========================
# COMBINED DEVELOPMENT LEADERBOARD
# =========================

combined_rows = []
recipe_registry: dict[str, dict[str, Any]] = {}

for _, row in development_leaderboard.iterrows():
    key = row["key"]
    combined_rows.append({
        "key": key,
        "recipe_type": "standalone",
        **row.to_dict(),
    })
    recipe_registry[key] = development_specs[key]

for _, row in blend_leaderboard.iterrows():
    key = row["key"]
    combined_rows.append({
        "key": key,
        "recipe_type": "blend",
        **row.to_dict(),
    })
    recipe_registry[key] = blend_recipes[key]

for _, row in specialist_leaderboard.iterrows():
    if "adopted_by_guard" not in row or not bool(row["adopted_by_guard"]):
        continue
    key = row["key"]
    combined_rows.append({
        "key": key,
        "recipe_type": "ro_specialist",
        **row.to_dict(),
    })
    recipe_registry[key] = specialist_recipes[key]

combined_development = (
    pd.DataFrame(combined_rows)
    .sort_values(["macro_f1", "errors"], ascending=[False, True])
    .drop_duplicates(subset=["key"])
    .reset_index(drop=True)
)
combined_development.to_csv(
    OUTPUT_DIR / "development_combined_leaderboard.csv", index=False
)
display(combined_development.head(30))

selected_recipe_keys = combined_development.head(
    PROFILE["assessment_recipes"]
)["key"].tolist()

control_spec = {
    "type": "standalone",
    "model": "torch_mlp512",
    "readout": "final_concat",
    "purpose": "head_zoo",
}
control_key = spec_key(control_spec)
recipe_registry[control_key] = control_spec

if control_key not in selected_recipe_keys:
    selected_recipe_keys.append(control_key)

locked_recipes = {
    key: recipe_registry[key]
    for key in selected_recipe_keys
}

atomic_json_dump(
    {
        "selection_source": "development_fold_0_only",
        "manual_test_labels_accessed": False,
        "selected_keys": selected_recipe_keys,
        "recipes": locked_recipes,
    },
    OUTPUT_DIR / "locked_recipes_before_assessment.json",
)

print("Locked recipes:")
for key, recipe in locked_recipes.items():
    print(key, recipe)

,key,recipe_type,model,readout,dimension,rows,macro_f1,accuracy,log_loss,errors,...,f1_electronic,f1_organic,recyclable_to_organic,organic_to_recyclable,ro_errors,training_seconds,left_key,right_key,alpha_left,disagreement_rate
0,8ed362eb569205b9,standalone,torch_msd_mlp512,mean_last4_concat,8192.0,5306,0.990491,0.988881,0.094632,59,...,0.996849,0.988827,20,34,54,13.180415,NaN,NaN,NaN,NaN
1,35e966fbb1c1dfd7,blend,NaN,NaN,NaN,5306,0.990491,0.988881,0.095031,59,...,0.996849,0.988827,20,34,54,NaN,8ed362eb569205b9,85a24bec6d8a75c5,0.9,0.004146
2,2b8161d1788b0756,blend,NaN,NaN,NaN,5306,0.990342,0.988692,0.094587,60,...,0.996849,0.988625,20,35,55,NaN,8ed362eb569205b9,dceb88081c8e8773,0.7,0.001696
3,45f9d06646bb925f,blend,NaN,NaN,NaN,5306,0.990342,0.988692,0.094729,60,...,0.996849,0.988625,20,35,55,NaN,8ed362eb569205b9,e03f9d1980422fa9,0.7,0.003015
4,cbc292d6bd324943,blend,NaN,NaN,NaN,5306,0.990211,0.988692,0.100478,60,...,0.996222,0.988643,24,30,54,NaN,85a24bec6d8a75c5,5e568f93459356bc,0.9,0.004900
5,8180b55511a924b2,blend,NaN,NaN,NaN,5306,0.990199,0.988692,0.097749,60,...,0.996217,0.988822,19,35,54,NaN,dceb88081c8e8773,85a24bec6d8a75c5,0.5,0.004712
6,70f80e2d5bd03bb6,blend,NaN,NaN,NaN,5306,0.990193,0.988504,0.094212,61,...,0.996849,0.988423,20,36,56,NaN,8ed362eb569205b9,8eff61d9aa46ceb6,0.8,0.002073
7,aff449bcc6dcfc07,blend,NaN,NaN,NaN,5306,0.990193,0.988504,0.094550,61,...,0.996849,0.988423,20,36,56,NaN,8ed362eb569205b9,5e568f93459356bc,0.9,0.001885
8,8eff61d9aa46ceb6,standalone,torch_mlp512,layer_mean_std_concat,16384.0,5306,0.990170,0.988315,0.093743,62,...,0.997478,0.988226,21,37,58,27.366078,NaN,NaN,NaN,NaN
9,bf378440bf7ae784,blend,NaN,NaN,NaN,5306,0.990068,0.988692,0.100644,60,...,0.995589,0.988840,23,30,53,NaN,85a24bec6d8a75c5,e03f9d1980422fa9,0.9,0.003769


Locked recipes:
8ed362eb569205b9 {'type': 'standalone', 'model': 'torch_msd_mlp512', 'readout': 'mean_last4_concat', 'purpose': 'head_zoo'}
35e966fbb1c1dfd7 {'type': 'blend', 'left': {'type': 'standalone', 'model': 'torch_msd_mlp512', 'readout': 'mean_last4_concat', 'purpose': 'head_zoo'}, 'right': {'type': 'standalone', 'model': 'torch_mlp512', 'readout': 'final_concat', 'purpose': 'head_zoo'}, 'alpha_left': 0.9, 'purpose': 'pairwise_blend'}
2b8161d1788b0756 {'type': 'blend', 'left': {'type': 'standalone', 'model': 'torch_msd_mlp512', 'readout': 'mean_last4_concat', 'purpose': 'head_zoo'}, 'right': {'type': 'standalone', 'model': 'torch_mlp512', 'readout': 'mean_last4_concat', 'purpose': 'head_zoo'}, 'alpha_left': 0.7, 'purpose': 'pairwise_blend'}
85a24bec6d8a75c5 {'type': 'standalone', 'model': 'torch_mlp512', 'readout': 'final_concat', 'purpose': 'head_zoo'}


# Stage F — Assessment folds 1–4

Recipe sudah dikunci. Setiap standalone model dilatih ulang pada training partition masing-masing fold.

Prediction cache mencegah base model yang sama dilatih ulang ketika dipakai oleh beberapa blend atau specialist.

In [23]:
# =========================
# RECURSIVE RECIPE FITTER
# =========================

assessment_prediction_cache: dict[tuple[int, str], np.ndarray] = {}
assessment_metadata_cache: dict[tuple[int, str], dict[str, Any]] = {}

def fit_recipe(
    recipe: dict[str, Any],
    train_indices: np.ndarray,
    valid_indices: np.ndarray,
    fold: int,
) -> tuple[np.ndarray, dict[str, Any]]:
    key = spec_key(recipe)
    cache_key = (fold, key)
    disk_path = PRED_DIR / f"assessment_fold{fold}_{key}.npy"
    meta_path = OUTPUT_DIR / f"assessment_fold{fold}_{key}.json"

    if cache_key in assessment_prediction_cache:
        return (
            assessment_prediction_cache[cache_key],
            assessment_metadata_cache[cache_key],
        )

    if RESUME and disk_path.exists() and meta_path.exists():
        probabilities = np.load(disk_path)
        metadata = json.loads(meta_path.read_text())
        assessment_prediction_cache[cache_key] = probabilities
        assessment_metadata_cache[cache_key] = metadata
        return probabilities, metadata

    if recipe["type"] == "standalone":
        probabilities, base_metadata = fit_predict_standalone(
            recipe, train_indices, valid_indices, fold
        )
        metadata = {
            "type": "standalone",
            "base_metadata": base_metadata,
        }

    elif recipe["type"] == "blend":
        left_prob, left_meta = fit_recipe(
            recipe["left"], train_indices, valid_indices, fold
        )
        right_prob, right_meta = fit_recipe(
            recipe["right"], train_indices, valid_indices, fold
        )
        alpha = float(recipe["alpha_left"])
        probabilities = (
            alpha * left_prob + (1.0 - alpha) * right_prob
        ).astype(np.float32)
        probabilities /= probabilities.sum(axis=1, keepdims=True)
        metadata = {
            "type": "blend",
            "alpha_left": alpha,
            "left_key": spec_key(recipe["left"]),
            "right_key": spec_key(recipe["right"]),
            "left_metadata": left_meta,
            "right_metadata": right_meta,
        }

    elif recipe["type"] == "ro_specialist":
        main_prob, main_meta = fit_recipe(
            recipe["main"], train_indices, valid_indices, fold
        )
        p_organic, specialist_meta = fit_predict_binary_specialist(
            recipe["specialist_model"],
            recipe["specialist_readout"],
            train_indices,
            valid_indices,
            fold,
        )
        probabilities = fuse_ro_specialist(
            main_prob,
            p_organic,
            float(recipe["alpha"]),
        )
        metadata = {
            "type": "ro_specialist",
            "main_key": spec_key(recipe["main"]),
            "main_metadata": main_meta,
            "specialist_model": recipe["specialist_model"],
            "specialist_readout": recipe["specialist_readout"],
            "specialist_metadata": specialist_meta,
            "alpha": float(recipe["alpha"]),
        }

    else:
        raise KeyError(recipe["type"])

    np.save(disk_path, probabilities)
    atomic_json_dump(metadata, meta_path)
    assessment_prediction_cache[cache_key] = probabilities
    assessment_metadata_cache[cache_key] = metadata
    return probabilities, metadata

In [24]:
# =========================
# RUN LOCKED ASSESSMENT
# =========================

assessment_rows = []
oof_probabilities = {
    key: np.full((EXPECTED_ROWS, NUM_CLASSES), np.nan, dtype=np.float32)
    for key in locked_recipes
}

for fold in ASSESSMENT_FOLDS:
    train_indices = np.flatnonzero(fold_array != fold)
    valid_indices = np.flatnonzero(fold_array == fold)
    valid_labels = cached_labels[torch.as_tensor(valid_indices)].numpy()

    print("\n" + "=" * 80)
    print("ASSESSMENT FOLD", fold)
    print("=" * 80)

    for key, recipe in locked_recipes.items():
        probabilities, metadata = fit_recipe(
            recipe,
            train_indices,
            valid_indices,
            fold,
        )
        metrics = metric_bundle(valid_labels, probabilities)
        oof_probabilities[key][valid_indices] = probabilities

        assessment_rows.append({
            "key": key,
            "recipe_type": recipe["type"],
            "fold": fold,
            **{k: v for k, v in metrics.items() if not isinstance(v, (dict, list))},
        })

        save_candidate_result(
            recipe,
            valid_indices,
            probabilities,
            metrics,
            stage="assessment",
            fold=fold,
        )

assessment_per_fold = pd.DataFrame(assessment_rows)
assessment_per_fold.to_csv(
    OUTPUT_DIR / "assessment_per_fold.csv", index=False
)
display(
    assessment_per_fold.sort_values(
        ["fold", "macro_f1"], ascending=[True, False]
    )
)


ASSESSMENT FOLD 1

Candidate 8ed362eb569205b9: model=torch_msd_mlp512, readout=mean_last4_concat, train=(21221, 8192), valid=(5306, 8192)
torch_msd_mlp512 fold=1 epoch=01 loss=0.280788 f1=0.987130
torch_msd_mlp512 fold=1 epoch=02 loss=0.226068 f1=0.988287
torch_msd_mlp512 fold=1 epoch=03 loss=0.218389 f1=0.988859
torch_msd_mlp512 fold=1 epoch=04 loss=0.215204 f1=0.988429
torch_msd_mlp512 fold=1 epoch=05 loss=0.213418 f1=0.989079
torch_msd_mlp512 fold=1 epoch=06 loss=0.212528 f1=0.988755
torch_msd_mlp512 fold=1 epoch=07 loss=0.212092 f1=0.989474
torch_msd_mlp512 fold=1 epoch=08 loss=0.211792 f1=0.989181
torch_msd_mlp512 fold=1 epoch=09 loss=0.211607 f1=0.989206
torch_msd_mlp512 fold=1 epoch=10 loss=0.211358 f1=0.989206
torch_msd_mlp512 fold=1 epoch=11 loss=0.211296 f1=0.989331
torch_msd_mlp512 fold=1 epoch=12 loss=0.211101 f1=0.989331

Candidate 85a24bec6d8a75c5: model=torch_mlp512, readout=final_concat, train=(21221, 8192), valid=(5306, 8192)
torch_mlp512 fold=1 epoch=01 loss=0.285692

,key,recipe_type,fold,rows,macro_f1,accuracy,log_loss,errors,f1_recyclable,f1_electronic,f1_organic,recyclable_to_organic,organic_to_recyclable,ro_errors
3,85a24bec6d8a75c5,standalone,1,5306,0.990363,0.989446,0.093566,56,0.986513,0.994343,0.990233,19,28,47
1,35e966fbb1c1dfd7,blend,1,5306,0.989749,0.988692,0.092791,60,0.985276,0.994343,0.989629,19,32,51
0,8ed362eb569205b9,standalone,1,5306,0.989474,0.988504,0.092854,61,0.985276,0.993719,0.989427,19,32,51
2,2b8161d1788b0756,blend,1,5306,0.988898,0.987938,0.092459,64,0.984769,0.993095,0.988831,21,32,53
4,8ed362eb569205b9,standalone,2,5305,0.991423,0.990763,0.088256,49,0.988511,0.994322,0.991436,18,22,40
5,35e966fbb1c1dfd7,blend,2,5305,0.991423,0.990763,0.087720,49,0.988511,0.994322,0.991436,18,22,40
6,2b8161d1788b0756,blend,2,5305,0.991422,0.990763,0.087278,49,0.988506,0.994322,0.991439,19,21,40
7,85a24bec6d8a75c5,standalone,2,5305,0.990982,0.990386,0.085028,51,0.988030,0.993687,0.991228,16,25,41
10,2b8161d1788b0756,blend,3,5305,0.990033,0.988690,0.096160,60,0.985283,0.995578,0.989239,22,31,53
8,8ed362eb569205b9,standalone,3,5305,0.989750,0.988313,0.097247,62,0.985022,0.995584,0.988643,24,31,55


In [25]:
# =========================
# AGGREGATE ASSESSMENT OOF
# =========================

assessment_indices = np.flatnonzero(
    np.isin(fold_array, np.asarray(ASSESSMENT_FOLDS))
)
assessment_labels = cached_labels[
    torch.as_tensor(assessment_indices)
].numpy()

aggregate_rows = []

for key, recipe in locked_recipes.items():
    probabilities = oof_probabilities[key][assessment_indices]
    if not np.isfinite(probabilities).all():
        raise RuntimeError(f"Incomplete assessment OOF for {key}")

    metrics = metric_bundle(assessment_labels, probabilities)
    per_fold_subset = assessment_per_fold[
        assessment_per_fold["key"] == key
    ]

    aggregate_rows.append({
        "key": key,
        "recipe_type": recipe["type"],
        "folds_won_against_control": None,
        "mean_fold_macro_f1": per_fold_subset["macro_f1"].mean(),
        "std_fold_macro_f1": per_fold_subset["macro_f1"].std(ddof=0),
        **{k: v for k, v in metrics.items() if not isinstance(v, (dict, list))},
    })

    prediction_frame(
        assessment_indices,
        probabilities,
        key,
        fold=-1,
        stage="assessment_aggregate",
    ).to_csv(
        PRED_DIR / f"assessment_aggregate_{key}.csv",
        index=False,
    )

assessment_aggregate = (
    pd.DataFrame(aggregate_rows)
    .sort_values(["macro_f1", "errors"], ascending=[False, True])
    .reset_index(drop=True)
)

# Compare every recipe with the V7-style control fold by fold.
control_per_fold = assessment_per_fold[
    assessment_per_fold["key"] == control_key
].set_index("fold")["macro_f1"]

for row_index, row in assessment_aggregate.iterrows():
    candidate_per_fold = assessment_per_fold[
        assessment_per_fold["key"] == row["key"]
    ].set_index("fold")["macro_f1"]
    wins = int((candidate_per_fold > control_per_fold).sum())
    assessment_aggregate.loc[row_index, "folds_won_against_control"] = wins

assessment_aggregate.to_csv(
    OUTPUT_DIR / "assessment_aggregate.csv", index=False
)
display(assessment_aggregate)

,key,recipe_type,folds_won_against_control,mean_fold_macro_f1,std_fold_macro_f1,rows,macro_f1,accuracy,log_loss,errors,f1_recyclable,f1_electronic,f1_organic,recyclable_to_organic,organic_to_recyclable,ro_errors
0,35e966fbb1c1dfd7,blend,2,0.990440,0.000721,21221,0.990439,0.989397,0.092248,225,0.986577,0.994804,0.989937,82,110,192
1,8ed362eb569205b9,standalone,2,0.990333,0.000773,21221,0.990333,0.989303,0.092392,227,0.986513,0.994647,0.989838,83,110,193
2,85a24bec6d8a75c5,standalone,0,0.990301,0.000780,21221,0.990301,0.989397,0.092814,225,0.986645,0.994174,0.990083,76,112,188
3,2b8161d1788b0756,blend,2,0.990223,0.000912,21221,0.990222,0.989209,0.091592,229,0.986389,0.994489,0.989788,84,110,194


# Final train-only decision

Recipe final ditentukan dari aggregate Macro F1 folds 1–4. Manual test labels tetap tidak disentuh.

Untuk mengurangi risiko memilih noise, notebook juga mencatat:

- standard deviation antar-fold,
- jumlah fold yang mengalahkan V7-style control,
- total errors,
- R/O errors,
- Electronic F1.

In [26]:
# =========================
# FINAL TRAIN-ONLY DECISION
# =========================

best_row = assessment_aggregate.iloc[0]
best_key = best_row["key"]
best_recipe = locked_recipes[best_key]

control_row = assessment_aggregate[
    assessment_aggregate["key"] == control_key
].iloc[0]

decision = {
    "selection_basis": "train_only_cross_validation",
    "development_fold": DEVELOPMENT_FOLD,
    "assessment_folds": list(ASSESSMENT_FOLDS),
    "manual_test_labels_accessed": False,
    "test_images_accessed": False,
    "best_key": best_key,
    "best_recipe": best_recipe,
    "best_metrics": best_row.to_dict(),
    "control_key": control_key,
    "control_recipe": control_spec,
    "control_metrics": control_row.to_dict(),
    "macro_f1_gain_over_control": float(
        best_row["macro_f1"] - control_row["macro_f1"]
    ),
    "adoption_recommendation": bool(
        best_key != control_key
        and best_row["macro_f1"] > control_row["macro_f1"]
        and best_row["errors"] <= control_row["errors"]
        and best_row["folds_won_against_control"] >= 3
    ),
    "next_step": (
        "Use this locked recipe in a separate automated test-inference notebook. "
        "Do not inspect or use test labels."
    ),
}

atomic_json_dump(decision, OUTPUT_DIR / "final_train_only_decision.json")
print(json.dumps(decision, indent=2, default=str))

{
  "selection_basis": "train_only_cross_validation",
  "development_fold": 0,
  "assessment_folds": [
    1,
    2,
    3,
    4
  ],
  "manual_test_labels_accessed": false,
  "test_images_accessed": false,
  "best_key": "35e966fbb1c1dfd7",
  "best_recipe": {
    "type": "blend",
    "left": {
      "type": "standalone",
      "model": "torch_msd_mlp512",
      "readout": "mean_last4_concat",
      "purpose": "head_zoo"
    },
    "right": {
      "type": "standalone",
      "model": "torch_mlp512",
      "readout": "final_concat",
      "purpose": "head_zoo"
    },
    "alpha_left": 0.9,
    "purpose": "pairwise_blend"
  },
  "best_metrics": {
    "key": "35e966fbb1c1dfd7",
    "recipe_type": "blend",
    "folds_won_against_control": 2,
    "mean_fold_macro_f1": 0.9904395685925576,
    "std_fold_macro_f1": 0.000720562511214698,
    "rows": 21221,
    "macro_f1": 0.9904393207699855,
    "accuracy": 0.9893972951321803,
    "log_loss": 0.09224824081721991,
    "errors": 225,
    "f1_rec

In [27]:
# =========================
# ARTIFACT INVENTORY
# =========================

artifact_rows = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        artifact_rows.append({
            "path": str(path.relative_to(OUTPUT_DIR)),
            "size_bytes": path.stat().st_size,
        })

artifact_inventory = pd.DataFrame(artifact_rows)
artifact_inventory.to_csv(OUTPUT_DIR / "artifact_inventory.csv", index=False)
display(artifact_inventory.tail(30))

print("Completed. Output directory:", OUTPUT_DIR)

,path,size_bytes
118,predictions/development_d0e7fe7416833908.npy,63800
119,predictions/development_dceb88081c8e8773.npy,63800
120,predictions/development_e03f9d1980422fa9.npy,63800
121,predictions/development_efc513d6834ff7d6.npy,63800
122,predictions/development_f54f4106b79328bd.npy,63800
123,predictions/readout_screen_2254c9f290025dfa.npy,63800
124,predictions/readout_screen_2914bd13c27211ad.npy,63800
125,predictions/readout_screen_2a54c3475f63a7da.npy,63800
126,predictions/readout_screen_3944aceabc08ac3e.npy,63800
127,predictions/readout_screen_56510518a3fa7085.npy,63800


Completed. Output directory: /kaggle/working/v9a_v7_cached_head_zoo


## Interpretation guide

Model yang paling tinggi di fold 0 belum otomatis diadopsi. Keputusan akhir memakai folds 1–4 setelah recipe dikunci.

Recipe baru layak menggantikan control ketika:

1. aggregate Macro F1 lebih tinggi,
2. total error tidak bertambah,
3. menang atas control pada minimal 3 dari 4 fold,
4. Electronic F1 tetap stabil,
5. gain tidak hanya berasal dari satu fold.

`manual-labelling.csv` tidak boleh digunakan untuk memutuskan hasil selama kompetisi. Setelah kompetisi benar-benar selesai, evaluasi test ground truth hanya boleh dilakukan sebagai post-competition research terpisah dan tidak boleh memengaruhi submission kompetisi.